# Soraya Colab Launcher

This notebook runs the Soraya Gradio MVP inside Google Colab without Docker. It installs dependencies, securely prompts for your Anthropic API key, writes the app files into the Colab runtime, and launches a temporary Gradio share link.

Use this only for private testing. The Gradio share link is public to anyone who has the link, and the Colab runtime is temporary.

## Setup

Run each cell in order. When prompted, paste your Anthropic API key. The key is stored only in the current Colab runtime environment variable, not in the notebook file.

In [ ]:
!pip install -q anthropic gradio
print('Dependencies installed')


In [ ]:
import os
from getpass import getpass

if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Paste ANTHROPIC_API_KEY: ')
os.environ.setdefault('ANTHROPIC_MODEL', 'claude-sonnet-4-6')
print('ANTHROPIC_MODEL =', os.environ['ANTHROPIC_MODEL'])


## Write Soraya app files

This creates `router.py`, `prompts.py`, `agency_ledger.py`, and `app.py` in the Colab runtime.

In [ ]:
from pathlib import Path
Path('router.py').write_text('from dataclasses import dataclass, asdict\nfrom enum import Enum\nimport re\nfrom typing import Dict, List\n\n\nclass GovernanceRoute(str, Enum):\n    FAST_PATH = "FAST_PATH"\n    MEDIUM_PATH = "MEDIUM_PATH"\n    DEEP_C3_ICM_PATH = "DEEP_C3_ICM_PATH"\n\n\nclass SorayaMode(str, Enum):\n    CLARIFY = "clarify"\n    TRIAGE = "triage"\n    TINY_STEP = "tiny_step"\n    SAFETY = "safety"\n\n\n@dataclass\nclass RouterDecision:\n    selected_route: GovernanceRoute\n    soraya_mode: SorayaMode\n    cognitive_signals: Dict[str, float]\n    risk_signals: Dict[str, bool]\n    user_action_required: str\n    review_required: bool\n    rationale: List[str]\n    confidence: float\n\n    def to_dict(self) -> Dict:\n        return asdict(self)\n\n\nSAFETY_PATTERNS = {\n    "self_harm": [r"\\bkill myself\\b", r"\\bsuicide\\b", r"\\bend my life\\b", r"\\bhurt myself\\b", r"\\bself[- ]harm\\b"],\n    "violence": [r"\\bhurt someone\\b", r"\\battack\\b", r"\\bweapon\\b", r"\\bmake a bomb\\b"],\n    "medical": [r"\\bdiagnose\\b", r"\\bmedication\\b", r"\\bprescribe\\b", r"\\bsymptoms\\b", r"\\bdoctor\\b", r"\\btherapy\\b"],\n    "legal": [r"\\bsue\\b", r"\\blawsuit\\b", r"\\blegal advice\\b", r"\\bcontract\\b", r"\\battorney\\b"],\n    "financial": [r"\\binvest\\b", r"\\bstock\\b", r"\\bcrypto\\b", r"\\btaxes\\b", r"\\bloan\\b", r"\\bretirement account\\b"],\n    "hr_decisioning": [r"\\bfire this employee\\b", r"\\bhire this person\\b", r"\\bpromote\\b", r"\\bperformance review\\b", r"\\bdisciplinary action\\b"],\n}\n\nCOGNITIVE_PATTERNS = {\n    "ambiguity": [r"\\bi don\'t know where to start\\b", r"\\bwhere do i start\\b", r"\\bwhat should i do\\b", r"\\bnot sure what to do\\b", r"\\bconfused\\b", r"\\bunclear\\b"],\n    "overload": [r"\\boverwhelmed\\b", r"\\btoo much\\b", r"\\bso many things\\b", r"\\bi have a lot\\b", r"\\bcan\'t keep up\\b", r"\\beverything at once\\b"],\n    "initiation_friction": [r"\\bprocrastinating\\b", r"\\bcan\'t start\\b", r"\\bstuck\\b", r"\\bfrozen\\b", r"\\bavoiding\\b", r"\\bputting it off\\b"],\n    "emotional_drag": [r"\\bi feel stupid\\b", r"\\bi\'m failing\\b", r"\\bi hate this\\b", r"\\bfrustrated\\b", r"\\bdiscouraged\\b", r"\\bshame\\b"],\n    "dependency_risk": [r"\\bjust tell me what to do\\b", r"\\bdecide for me\\b", r"\\bdo it all for me\\b", r"\\bi need you to choose\\b", r"\\bwhat would you do if you were me\\b"],\n}\n\nMEDIUM_STAKES_PATTERNS = [\n    r"\\bmanager\\b", r"\\bworkplace\\b", r"\\bclient\\b", r"\\bcustomer\\b", r"\\bpolicy\\b",\n    r"\\bcompliance\\b", r"\\bschool assignment\\b", r"\\bteacher\\b", r"\\bdeadline\\b",\n    r"\\bresume\\b", r"\\binterview\\b",\n]\n\n\ndef normalize_text(text: str) -> str:\n    return text.lower().strip()\n\n\ndef pattern_match(text: str, patterns: List[str]) -> bool:\n    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)\n\n\ndef score_pattern_group(text: str, patterns: List[str]) -> float:\n    matches = sum(1 for pattern in patterns if re.search(pattern, text, flags=re.IGNORECASE))\n    if matches == 0:\n        return 0.0\n    if matches == 1:\n        return 0.5\n    return 1.0\n\n\ndef detect_risk_signals(user_text: str) -> Dict[str, bool]:\n    text = normalize_text(user_text)\n    risk_signals = {risk_name: pattern_match(text, patterns) for risk_name, patterns in SAFETY_PATTERNS.items()}\n    risk_signals["medium_stakes"] = pattern_match(text, MEDIUM_STAKES_PATTERNS)\n    return risk_signals\n\n\ndef detect_cognitive_signals(user_text: str) -> Dict[str, float]:\n    text = normalize_text(user_text)\n    return {signal_name: score_pattern_group(text, patterns) for signal_name, patterns in COGNITIVE_PATTERNS.items()}\n\n\ndef select_governance_route(risk_signals: Dict[str, bool]) -> GovernanceRoute:\n    deep_risks = ["self_harm", "violence", "medical", "legal", "financial", "hr_decisioning"]\n    if any(risk_signals.get(risk, False) for risk in deep_risks):\n        return GovernanceRoute.DEEP_C3_ICM_PATH\n    if risk_signals.get("medium_stakes", False):\n        return GovernanceRoute.MEDIUM_PATH\n    return GovernanceRoute.FAST_PATH\n\n\ndef select_soraya_mode(route: GovernanceRoute, cognitive_signals: Dict[str, float]) -> SorayaMode:\n    if route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        return SorayaMode.SAFETY\n    dependency = cognitive_signals.get("dependency_risk", 0.0)\n    overload = cognitive_signals.get("overload", 0.0)\n    ambiguity = cognitive_signals.get("ambiguity", 0.0)\n    initiation = cognitive_signals.get("initiation_friction", 0.0)\n    emotional = cognitive_signals.get("emotional_drag", 0.0)\n    if dependency >= 0.5:\n        return SorayaMode.CLARIFY\n    if overload >= 0.5:\n        return SorayaMode.TRIAGE\n    if emotional >= 0.5 and initiation >= 0.5:\n        return SorayaMode.TINY_STEP\n    if ambiguity >= 0.5:\n        return SorayaMode.CLARIFY\n    if initiation >= 0.5:\n        return SorayaMode.TINY_STEP\n    return SorayaMode.CLARIFY\n\n\ndef build_user_action_required(mode: SorayaMode, route: GovernanceRoute) -> str:\n    if route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        return "Pause and seek appropriate human, professional, or policy review before acting."\n    if mode == SorayaMode.CLARIFY:\n        return "Answer one clarifying question or choose one goal to work on."\n    if mode == SorayaMode.TRIAGE:\n        return "Choose the top priority from a shortened list."\n    if mode == SorayaMode.TINY_STEP:\n        return "Complete one small starter action."\n    return "Choose the next responsible action."\n\n\ndef build_rationale(route: GovernanceRoute, mode: SorayaMode, risk_signals: Dict[str, bool], cognitive_signals: Dict[str, float]) -> List[str]:\n    rationale = []\n    active_risks = [key for key, value in risk_signals.items() if value]\n    active_cognitive = [key for key, value in cognitive_signals.items() if value >= 0.5]\n    rationale.append(f"Selected governance route: {route.value}.")\n    rationale.append(f"Selected Soraya mode: {mode.value}.")\n    rationale.append(f"Detected risk/context signals: {\', \'.join(active_risks)}." if active_risks else "No high-risk domain signals detected.")\n    rationale.append(f"Detected executive-function signals: {\', \'.join(active_cognitive)}." if active_cognitive else "No strong executive-function distress signal detected; defaulting to clarification.")\n    return rationale\n\n\ndef estimate_confidence(route: GovernanceRoute, cognitive_signals: Dict[str, float], risk_signals: Dict[str, bool]) -> float:\n    if route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        return 0.85\n    max_cognitive = max(cognitive_signals.values()) if cognitive_signals else 0.0\n    if risk_signals.get("medium_stakes", False):\n        return 0.75\n    if max_cognitive >= 1.0:\n        return 0.8\n    if max_cognitive >= 0.5:\n        return 0.7\n    return 0.6\n\n\ndef route_user_turn(user_text: str) -> RouterDecision:\n    if not user_text or not user_text.strip():\n        return RouterDecision(\n            selected_route=GovernanceRoute.FAST_PATH,\n            soraya_mode=SorayaMode.CLARIFY,\n            cognitive_signals={"ambiguity": 1.0, "overload": 0.0, "initiation_friction": 0.0, "emotional_drag": 0.0, "dependency_risk": 0.0},\n            risk_signals={"self_harm": False, "violence": False, "medical": False, "legal": False, "financial": False, "hr_decisioning": False, "medium_stakes": False},\n            user_action_required="Name the task or situation you want help with.",\n            review_required=False,\n            rationale=["Empty or unclear input; defaulting to Clarify Mode."],\n            confidence=0.6,\n        )\n    risk_signals = detect_risk_signals(user_text)\n    cognitive_signals = detect_cognitive_signals(user_text)\n    route = select_governance_route(risk_signals)\n    mode = select_soraya_mode(route, cognitive_signals)\n    return RouterDecision(\n        selected_route=route,\n        soraya_mode=mode,\n        cognitive_signals=cognitive_signals,\n        risk_signals=risk_signals,\n        user_action_required=build_user_action_required(mode, route),\n        review_required=route == GovernanceRoute.DEEP_C3_ICM_PATH,\n        rationale=build_rationale(route, mode, risk_signals, cognitive_signals),\n        confidence=estimate_confidence(route, cognitive_signals, risk_signals),\n    )\n', encoding='utf-8')
print('Wrote router.py')


In [ ]:
from pathlib import Path
Path('prompts.py').write_text('from router import SorayaMode, RouterDecision\n\n\nSORAYA_IDENTITY = """\nYou are Soraya, a human-centered AI coaching system built by Kaleidoworks.\n\nYour purpose is to strengthen executive function, not replace it.\n\nYou help people clarify, triage, start, and follow through.\nYou do not make decisions for users. You help users make better decisions for themselves.\nYou return meaningful choice and action ownership to the user at the end of every response.\n\nHard limits:\n- You are not a therapist, clinician, crisis counselor, legal advisor, financial advisor, or HR decision-maker.\n- If content involves self-harm, violence, medical diagnosis, legal advice, financial advice, or HR employment decisions, pause, name the boundary clearly, and redirect to appropriate human support.\n- Never encourage dependency. If a user asks you to decide everything, make the decision smaller and hand it back.\n- Never rubber-stamp AI-generated content as verified fact.\n""".strip()\n\n\nCLARIFY_SYSTEM = f"""\n{SORAYA_IDENTITY}\n\n--- ACTIVE MODE: CLARIFY ---\n\nThe user is experiencing ambiguity or dependency pressure.\nYour job:\n1. Acknowledge the situation briefly.\n2. Ask exactly one clarifying question, or name the likely goal and ask the user to confirm or correct it.\n3. Do not produce a plan, list, or full solution yet.\n4. End with the user holding a choice.\n\nTone: calm, direct, not therapist-y.\nLength: two to four sentences maximum.\n""".strip()\n\n\nTRIAGE_SYSTEM = f"""\n{SORAYA_IDENTITY}\n\n--- ACTIVE MODE: TRIAGE ---\n\nThe user is overwhelmed by too many open loops or competing priorities.\nYour job:\n1. Acknowledge the overload briefly.\n2. Ask the user to name everything on their plate, or work with what they already said.\n3. Compress the field to a maximum of three priorities.\n4. Present the top priority clearly and ask the user to confirm it before moving forward.\n5. Explicitly park lower-priority items so the user knows they are not lost.\n\nDo not solve everything at once. The user chooses which priority to act on.\n""".strip()\n\n\nTINY_STEP_SYSTEM = f"""\n{SORAYA_IDENTITY}\n\n--- ACTIVE MODE: TINY_STEP ---\n\nThe user is stuck, avoiding, frozen, or unable to start.\nYour job:\n1. Do not lecture about productivity or motivation.\n2. Give exactly one small, concrete, completable action that takes five minutes or less.\n3. Name the action specifically.\n4. Tell the user what to do after the tiny step.\n\nTone: matter-of-fact and warm.\nLength: three to five sentences maximum.\n""".strip()\n\n\nSAFETY_SYSTEM = f"""\n{SORAYA_IDENTITY}\n\n--- ACTIVE MODE: SAFETY ---\n\nThe routing system detected content outside Soraya\'s coaching scope.\nYour job:\n1. Do not coach through the restricted domain.\n2. Name the boundary clearly and without judgment.\n3. For self-harm or crisis content, provide 988 Suicide and Crisis Lifeline directly: call or text 988.\n4. For legal, financial, medical, or HR domains, state what Soraya cannot do and offer to help prepare questions for the appropriate professional or human reviewer.\n5. End with a clear, low-friction next step.\n""".strip()\n\n\nUSER_TEMPLATE = """\nRouting context (do not repeat this to the user verbatim):\n- Governance route: {route}\n- Detected signals: {signals}\n- User action required: {action}\n\nUser message:\n{user_text}\n""".strip()\n\n\ndef build_prompt(decision: RouterDecision, user_text: str) -> dict:\n    mode = decision.soraya_mode\n    route = decision.selected_route.value\n\n    active_cognitive = [k for k, v in decision.cognitive_signals.items() if v >= 0.5]\n    active_risk = [k for k, v in decision.risk_signals.items() if v]\n    signals = active_cognitive + active_risk\n    signals_str = ", ".join(signals) if signals else "none detected"\n\n    if mode == SorayaMode.TRIAGE:\n        system_prompt = TRIAGE_SYSTEM\n    elif mode == SorayaMode.TINY_STEP:\n        system_prompt = TINY_STEP_SYSTEM\n    elif mode == SorayaMode.SAFETY:\n        system_prompt = SAFETY_SYSTEM\n    else:\n        system_prompt = CLARIFY_SYSTEM\n\n    return {\n        "system_prompt": system_prompt,\n        "user_message": USER_TEMPLATE.format(\n            route=route,\n            signals=signals_str,\n            action=decision.user_action_required,\n            user_text=user_text,\n        ),\n        "mode": mode.value,\n        "route": route,\n    }\n', encoding='utf-8')
print('Wrote prompts.py')


In [ ]:
from pathlib import Path
Path('agency_ledger.py').write_text('from dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom typing import Any, Dict, List, Optional\n\nfrom router import GovernanceRoute, SorayaMode, RouterDecision\n\n\nLEDGER_SCHEMA_VERSION = "soraya_agency_ledger_v0.1"\n\nMODE_SYSTEM_ACTIONS = {\n    SorayaMode.CLARIFY: "Returned ambiguity to the user as one smaller choice or clarifying question.",\n    SorayaMode.TRIAGE: "Compressed overload into a short priority frame while leaving final priority ownership with the user.",\n    SorayaMode.TINY_STEP: "Provided one small starter action to restore motion without taking over the task.",\n    SorayaMode.SAFETY: "Paused coaching and redirected toward appropriate human, professional, or policy review.",\n}\n\nMODE_NEXT_STEP_SIZE = {\n    SorayaMode.CLARIFY: "small",\n    SorayaMode.TRIAGE: "medium",\n    SorayaMode.TINY_STEP: "small",\n    SorayaMode.SAFETY: "small",\n}\n\nREVIEW_DOMAINS = {\n    "self_harm": "crisis_or_clinical_support",\n    "violence": "safety_or_emergency_support",\n    "medical": "medical_professional",\n    "legal": "legal_professional",\n    "financial": "financial_professional",\n    "hr_decisioning": "hr_or_manager_review",\n    "medium_stakes": "human_review_optional",\n}\n\n\n@dataclass\nclass AgencyLedgerEntry:\n    schema_version: str\n    turn_id: int\n    timestamp_utc: str\n    user_text_sha256: str\n    user_text_excerpt: Optional[str]\n    user_goal: str\n    current_context: str\n    selected_route: str\n    soraya_mode: str\n    cognitive_signals: Dict[str, float]\n    risk_signals: Dict[str, bool]\n    router_confidence: float\n    router_rationale: List[str]\n    user_action_required: str\n    system_action_taken: str\n    next_step_size: str\n    dependency_risk_score: float\n    user_agency_score: float\n    agency_balance_score: float\n    agency_delta_estimate: str\n    review_required: bool\n    review_domain: Optional[str]\n    evidence_needed: bool\n    evidence_note: str\n    storage_policy: str = (\n        "MVP ledger is intended for ephemeral session display. "\n        "Do not persist sensitive user data unless explicit storage policy is configured."\n    )\n\n    def to_dict(self) -> Dict[str, Any]:\n        return asdict(self)\n\n    def to_json(self, indent: int = 2) -> str:\n        return json.dumps(self.to_dict(), indent=indent, ensure_ascii=False)\n\n\n@dataclass\nclass AgencyLedger:\n    entries: List[AgencyLedgerEntry] = field(default_factory=list)\n\n    def add_turn(\n        self,\n        decision: RouterDecision,\n        user_text: str,\n        system_action_taken: Optional[str] = None,\n        user_goal: Optional[str] = None,\n        current_context: Optional[str] = None,\n        store_text_excerpt: bool = False,\n    ) -> AgencyLedgerEntry:\n        entry = create_ledger_entry(\n            decision=decision,\n            user_text=user_text,\n            turn_id=len(self.entries) + 1,\n            system_action_taken=system_action_taken,\n            user_goal=user_goal,\n            current_context=current_context,\n            store_text_excerpt=store_text_excerpt,\n        )\n        self.entries.append(entry)\n        return entry\n\n    def to_dict(self) -> Dict[str, Any]:\n        return {\n            "schema_version": LEDGER_SCHEMA_VERSION,\n            "turn_count": len(self.entries),\n            "summary": summarize_ledger([entry.to_dict() for entry in self.entries]),\n            "entries": [entry.to_dict() for entry in self.entries],\n        }\n\n    def to_json(self, indent: int = 2) -> str:\n        return json.dumps(self.to_dict(), indent=indent, ensure_ascii=False)\n\n    def clear(self) -> None:\n        self.entries.clear()\n\n\ndef create_ledger_entry(\n    decision: RouterDecision,\n    user_text: str,\n    turn_id: int = 1,\n    system_action_taken: Optional[str] = None,\n    user_goal: Optional[str] = None,\n    current_context: Optional[str] = None,\n    store_text_excerpt: bool = False,\n) -> AgencyLedgerEntry:\n    mode = decision.soraya_mode\n    route = decision.selected_route\n    dependency_risk_score = estimate_dependency_risk(decision)\n    user_agency_score = estimate_user_agency(decision, dependency_risk_score)\n    agency_balance_score = round(user_agency_score - dependency_risk_score, 3)\n    review_domain = select_review_domain(decision.risk_signals)\n    evidence_needed, evidence_note = estimate_evidence_need(decision)\n\n    return AgencyLedgerEntry(\n        schema_version=LEDGER_SCHEMA_VERSION,\n        turn_id=turn_id,\n        timestamp_utc=datetime.now(timezone.utc).isoformat(),\n        user_text_sha256=hash_text(user_text),\n        user_text_excerpt=make_excerpt(user_text) if store_text_excerpt else None,\n        user_goal=user_goal or infer_user_goal(user_text, decision),\n        current_context=current_context or infer_current_context(decision),\n        selected_route=route.value,\n        soraya_mode=mode.value,\n        cognitive_signals=round_signal_map(decision.cognitive_signals),\n        risk_signals=dict(decision.risk_signals),\n        router_confidence=round(float(decision.confidence), 3),\n        router_rationale=list(decision.rationale),\n        user_action_required=decision.user_action_required,\n        system_action_taken=system_action_taken or MODE_SYSTEM_ACTIONS.get(mode, "Provided bounded executive-function support."),\n        next_step_size=MODE_NEXT_STEP_SIZE.get(mode, "small"),\n        dependency_risk_score=dependency_risk_score,\n        user_agency_score=user_agency_score,\n        agency_balance_score=agency_balance_score,\n        agency_delta_estimate=estimate_agency_delta(user_agency_score, dependency_risk_score, decision.review_required),\n        review_required=bool(decision.review_required),\n        review_domain=review_domain,\n        evidence_needed=evidence_needed,\n        evidence_note=evidence_note,\n    )\n\n\ndef estimate_dependency_risk(decision: RouterDecision) -> float:\n    cognitive = decision.cognitive_signals\n    score = 0.0\n    score += 0.70 * cognitive.get("dependency_risk", 0.0)\n    score += 0.10 * cognitive.get("overload", 0.0)\n    score += 0.10 * cognitive.get("emotional_drag", 0.0)\n    score += 0.10 * cognitive.get("initiation_friction", 0.0)\n    return clamp(round(score, 3), 0.0, 1.0)\n\n\ndef estimate_user_agency(decision: RouterDecision, dependency_risk_score: float) -> float:\n    mode = decision.soraya_mode\n    route = decision.selected_route\n    score = 0.70\n    if mode == SorayaMode.CLARIFY:\n        score += 0.12\n    elif mode == SorayaMode.TRIAGE:\n        score += 0.08\n    elif mode == SorayaMode.TINY_STEP:\n        score += 0.10\n    elif mode == SorayaMode.SAFETY:\n        score += 0.02\n    if route == GovernanceRoute.MEDIUM_PATH:\n        score -= 0.03\n    elif route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        score -= 0.08\n    score -= 0.25 * dependency_risk_score\n    return clamp(round(score, 3), 0.0, 1.0)\n\n\ndef estimate_agency_delta(user_agency_score: float, dependency_risk_score: float, review_required: bool) -> str:\n    balance = user_agency_score - dependency_risk_score\n    if review_required:\n        return "watch_review_required"\n    if balance >= 0.55:\n        return "positive_agency_preserved"\n    if balance >= 0.30:\n        return "watch_agency_pressure"\n    return "risk_dependency_or_displacement"\n\n\ndef select_review_domain(risk_signals: Dict[str, bool]) -> Optional[str]:\n    for key in ["self_harm", "violence", "medical", "legal", "financial", "hr_decisioning", "medium_stakes"]:\n        if risk_signals.get(key, False):\n            return REVIEW_DOMAINS.get(key)\n    return None\n\n\ndef estimate_evidence_need(decision: RouterDecision) -> tuple[bool, str]:\n    if decision.selected_route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        return True, "Restricted or high-stakes content detected; appropriate human/professional review is needed."\n    if decision.selected_route == GovernanceRoute.MEDIUM_PATH:\n        return True, "Medium-stakes context detected; claims, policies, deadlines, or stakeholder impacts may need verification."\n    return False, "No explicit evidence requirement detected for this low-risk support turn."\n\n\ndef infer_user_goal(user_text: str, decision: RouterDecision) -> str:\n    text = (user_text or "").lower()\n    if "assignment" in text or "school" in text or "teacher" in text:\n        return "make progress on a learning or school task"\n    if "resume" in text or "interview" in text:\n        return "make progress on a career task"\n    if decision.soraya_mode == SorayaMode.TRIAGE:\n        return "reduce overload and identify the next priority"\n    if decision.soraya_mode == SorayaMode.TINY_STEP:\n        return "restore motion on a stuck task"\n    if decision.soraya_mode == SorayaMode.SAFETY:\n        return "handle a restricted or high-stakes request responsibly"\n    return "clarify the next responsible action"\n\n\ndef infer_current_context(decision: RouterDecision) -> str:\n    if decision.selected_route == GovernanceRoute.DEEP_C3_ICM_PATH:\n        return "high-stakes or restricted domain"\n    if decision.selected_route == GovernanceRoute.MEDIUM_PATH:\n        return "medium-stakes task context"\n    return "low-risk executive-function support"\n\n\ndef summarize_ledger(entry_dicts: List[Dict[str, Any]]) -> Dict[str, Any]:\n    if not entry_dicts:\n        return {"turn_count": 0}\n    n = len(entry_dicts)\n    return {\n        "turn_count": n,\n        "average_dependency_risk": round(sum(e["dependency_risk_score"] for e in entry_dicts) / n, 3),\n        "average_user_agency_score": round(sum(e["user_agency_score"] for e in entry_dicts) / n, 3),\n        "review_required_count": sum(1 for e in entry_dicts if e["review_required"]),\n    }\n\n\ndef hash_text(text: str) -> str:\n    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()\n\n\ndef make_excerpt(text: str, max_chars: int = 160) -> str:\n    clean = " ".join((text or "").split())\n    if len(clean) <= max_chars:\n        return clean\n    return clean[: max_chars - 1].rstrip() + "…"\n\n\ndef round_signal_map(signals: Dict[str, float]) -> Dict[str, float]:\n    return {key: round(float(value), 3) for key, value in signals.items()}\n\n\ndef clamp(value: float, low: float, high: float) -> float:\n    return max(low, min(high, value))\n\n\ndef format_entry_for_panel(entry) -> str:\n    return (\n        "### Agency Ledger Entry\\n\\n"\n        f"- **Turn:** `{entry.turn_id}`\\n"\n        f"- **Route:** `{entry.selected_route}`\\n"\n        f"- **Mode:** `{entry.soraya_mode}`\\n"\n        f"- **Dependency risk:** `{entry.dependency_risk_score}`\\n"\n        f"- **User agency score:** `{entry.user_agency_score}`\\n"\n        f"- **Agency balance:** `{entry.agency_balance_score}`\\n"\n        f"- **Agency delta:** `{entry.agency_delta_estimate}`\\n"\n        f"- **Review required:** `{entry.review_required}`\\n"\n        f"- **Review domain:** `{entry.review_domain}`\\n"\n        f"- **Evidence needed:** `{entry.evidence_needed}`\\n\\n"\n        "```json\\n"\n        + entry.to_json(indent=2)\n        + "\\n```"\n    )\n', encoding='utf-8')
print('Wrote agency_ledger.py')


In [ ]:
from pathlib import Path
Path('app.py').write_text('import os\nimport traceback\n\nimport anthropic\nimport gradio as gr\n\nfrom router import route_user_turn\nfrom prompts import build_prompt\nfrom agency_ledger import create_ledger_entry, format_entry_for_panel\n\n\nMODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-6")\nMAX_TOKENS = 512\nSTORE_EXCERPT = True\n\nSORAYA_WELCOME = (\n    "Hi. I\'m Soraya.\\n\\n"\n    "I help with starting, structuring, and following through without doing the thinking for you.\\n\\n"\n    "What are you working on or trying to figure out?"\n)\n\n\ndef call_llm(system_prompt: str, user_message: str) -> str:\n    api_key = os.environ.get("ANTHROPIC_API_KEY", "")\n    if not api_key:\n        return (\n            "[Configuration error: ANTHROPIC_API_KEY is not set. "\n            "Add it as a Space Secret or local environment variable.]"\n        )\n\n    client = anthropic.Anthropic(api_key=api_key)\n    message = client.messages.create(\n        model=MODEL,\n        max_tokens=MAX_TOKENS,\n        system=system_prompt,\n        messages=[{"role": "user", "content": user_message}],\n    )\n    text_blocks = [\n        block.text\n        for block in message.content\n        if getattr(block, "type", None) == "text"\n    ]\n    return "\\n".join(text_blocks).strip() or "[No text response returned.]"\n\n\ndef format_routing_panel(decision, prompt_meta: dict) -> str:\n    active_cog = [k for k, v in decision.cognitive_signals.items() if v >= 0.5]\n    active_risk = [k for k, v in decision.risk_signals.items() if v]\n    cog_str = ", ".join(active_cog) if active_cog else "none"\n    risk_str = ", ".join(active_risk) if active_risk else "none"\n    rationale_lines = "\\n".join(f"  - {r}" for r in decision.rationale)\n    return (\n        "### Routing Decision\\n\\n"\n        f"- **Governance route:** `{decision.selected_route.value}`\\n"\n        f"- **Soraya mode:** `{prompt_meta[\'mode\']}`\\n"\n        f"- **Router confidence:** `{decision.confidence}`\\n\\n"\n        f"**Cognitive signals detected:** {cog_str}\\n\\n"\n        f"**Risk signals detected:** {risk_str}\\n\\n"\n        f"**Rationale:**\\n{rationale_lines}\\n\\n"\n        f"**User action required:**\\n> {decision.user_action_required}"\n    )\n\n\ndef format_session_summary(entry_dicts: list) -> str:\n    if not entry_dicts:\n        return "_No turns recorded yet._"\n    n = len(entry_dicts)\n    avg_dep = round(sum(e["dependency_risk_score"] for e in entry_dicts) / n, 3)\n    avg_agency = round(sum(e["user_agency_score"] for e in entry_dicts) / n, 3)\n    review_count = sum(1 for e in entry_dicts if e["review_required"])\n    mode_counts = {}\n    route_counts = {}\n    for e in entry_dicts:\n        mode_counts[e["soraya_mode"]] = mode_counts.get(e["soraya_mode"], 0) + 1\n        route_counts[e["selected_route"]] = route_counts.get(e["selected_route"], 0) + 1\n    return (\n        f"**Session summary: {n} turn(s)**\\n\\n"\n        f"- Avg dependency risk: `{avg_dep}`\\n"\n        f"- Avg user agency score: `{avg_agency}`\\n"\n        f"- Turns requiring review: `{review_count}`\\n"\n        f"- Mode counts: `{mode_counts}`\\n"\n        f"- Route counts: `{route_counts}`"\n    )\n\n\ndef handle_turn(user_text: str, chat_history: list, ledger_state: dict) -> tuple:\n    if not user_text or not user_text.strip():\n        return (\n            chat_history,\n            "_No input detected. Type something to begin._",\n            "_Waiting for first turn._",\n            "_No turns yet._",\n            ledger_state,\n            "",\n        )\n\n    stored_entries = ledger_state.get("entries", [])\n    turn_id = len(stored_entries) + 1\n\n    try:\n        decision = route_user_turn(user_text)\n        prompt_meta = build_prompt(decision, user_text)\n        soraya_response = call_llm(\n            system_prompt=prompt_meta["system_prompt"],\n            user_message=prompt_meta["user_message"],\n        )\n        entry = create_ledger_entry(\n            decision=decision,\n            user_text=user_text,\n            turn_id=turn_id,\n            store_text_excerpt=STORE_EXCERPT,\n        )\n        stored_entries = stored_entries + [entry.to_dict()]\n        ledger_state = {"entries": stored_entries}\n        routing_panel = format_routing_panel(decision, prompt_meta)\n        ledger_panel = format_entry_for_panel(entry)\n        session_summary = format_session_summary(stored_entries)\n        chat_history = chat_history + [\n            {"role": "user", "content": user_text},\n            {"role": "assistant", "content": soraya_response},\n        ]\n    except Exception as e:\n        error_msg = f"[Error during turn: {type(e).__name__}: {e}]"\n        traceback.print_exc()\n        chat_history = chat_history + [\n            {"role": "user", "content": user_text},\n            {"role": "assistant", "content": error_msg},\n        ]\n        routing_panel = "_Routing error. See Space logs._"\n        ledger_panel = "_Ledger error. See Space logs._"\n        session_summary = "_Session error._"\n\n    return chat_history, routing_panel, ledger_panel, session_summary, ledger_state, ""\n\n\ndef reset_session() -> tuple:\n    return (\n        [{"role": "assistant", "content": SORAYA_WELCOME}],\n        "_Routing metadata will appear here after your first message._",\n        "_Agency Ledger will appear here after your first message._",\n        "_No turns yet._",\n        {},\n        "",\n    )\n\n\nCSS = """\n.panel-box { background: #f8f8f8; border-radius: 8px; padding: 12px; }\nfooter { display: none !important; }\n"""\n\nwith gr.Blocks(title="Soraya — Executive Function Infrastructure") as demo:\n    ledger_state = gr.State({})\n\n    gr.Markdown(\n        "## Soraya\\n"\n        "**Executive Function Infrastructure** · Kaleidoworks · Sprint 1 MVP\\n\\n"\n        "_Soraya helps you start, structure, verify, and follow through. "\n        "It does not make decisions for you._\\n\\n"\n        "---"\n    )\n\n    with gr.Row():\n        with gr.Column(scale=3):\n            chatbot = gr.Chatbot(\n                value=[{"role": "assistant", "content": SORAYA_WELCOME}],\n                label="Soraya",\n                height=480,\n            )\n            with gr.Row():\n                user_input = gr.Textbox(\n                    placeholder="What are you working on or trying to figure out?",\n                    label="",\n                    lines=2,\n                    scale=5,\n                    container=False,\n                )\n                submit_btn = gr.Button("Send", variant="primary", scale=1)\n            reset_btn = gr.Button("Reset session", variant="secondary", size="sm")\n\n        with gr.Column(scale=2):\n            routing_panel = gr.Markdown(\n                "_Routing metadata will appear here after your first message._",\n                label="Routing Decision",\n                elem_classes=["panel-box"],\n            )\n            gr.Markdown("---")\n            ledger_panel = gr.Markdown(\n                "_Agency Ledger will appear here after your first message._",\n                label="Agency Ledger",\n                elem_classes=["panel-box"],\n            )\n            gr.Markdown("---")\n            session_summary = gr.Markdown("_No turns yet._", label="Session Summary")\n\n    gr.Markdown(\n        "\\n---\\n"\n        "_Soraya is a behavioral MVP built by Kaleidoworks. "\n        "It is not a therapist, clinician, crisis counselor, legal advisor, financial advisor, or HR decision-maker. "\n        "Do not enter confidential, sensitive, regulated, or personally identifying information into this demo. "\n        "If you are in crisis, please contact the 988 Suicide and Crisis Lifeline by calling or texting **988**._"\n    )\n\n    turn_outputs = [chatbot, routing_panel, ledger_panel, session_summary, ledger_state, user_input]\n\n    submit_btn.click(fn=handle_turn, inputs=[user_input, chatbot, ledger_state], outputs=turn_outputs)\n    user_input.submit(fn=handle_turn, inputs=[user_input, chatbot, ledger_state], outputs=turn_outputs)\n    reset_btn.click(fn=reset_session, inputs=[], outputs=turn_outputs)\n\n\nif __name__ == "__main__":\n    demo.launch(css=CSS, ssr_mode=False, share=True, debug=True)\n', encoding='utf-8')
print('Wrote app.py')


## Quick deterministic smoke test

This checks routing, prompt construction, and Agency Ledger formatting without calling Anthropic.

In [ ]:
from router import route_user_turn
from prompts import build_prompt
from agency_ledger import create_ledger_entry, format_entry_for_panel

cases = [
    "I don't know where to start with this assignment.",
    "I have too much to do and can't keep up.",
    "I'm stuck and procrastinating on my resume.",
    "Just tell me what to do and decide for me.",
    "Can you give me legal advice about this contract?",
]

for i, text in enumerate(cases, 1):
    decision = route_user_turn(text)
    prompt = build_prompt(decision, text)
    entry = create_ledger_entry(decision, text, turn_id=i, store_text_excerpt=True)
    panel = format_entry_for_panel(entry)
    assert prompt['mode'] == decision.soraya_mode.value
    assert entry.turn_id == i
    assert 'Agency Ledger Entry' in panel
    print(i, decision.selected_route.value, decision.soraya_mode.value, entry.agency_delta_estimate)


## Launch Soraya

Run the next cell. Colab should print a temporary `gradio.live` URL. Open that URL on your iPhone to test Soraya.

In [ ]:
!python app.py


## Private smoke test prompts

Use these in the Gradio UI:

1. `I don't know where to start with this assignment.`
2. `I have too much to do and can't keep up.`
3. `I'm stuck and procrastinating on my resume.`
4. `Just tell me what to do and decide for me.`
5. `Can you give me legal advice about this contract?`

Reminder: do not enter confidential, sensitive, regulated, or personally identifying information into the demo.